In [4]:
import pandas as pd
import numpy as np
import folium
import os
from folium.plugins import HeatMap
import geopandas as gpd

# Créer fichier csv cntenant une colonne 'meurtrier', 'latitute' et 'longitude'

In [150]:
years = range(2022,2026)
df_all = pd.concat(
    [pd.read_csv(f'../data/raw/feminicide_{year}.csv') for year in years],
    ignore_index=True
)

In [151]:
df_all = df_all.rename(columns=lambda x: x.strip().lower().replace(' ', '_'))

In [152]:
df_all.columns = (
    df_all.columns
    .str.normalize("NFKD")
    .str.encode("ascii", "ignore")
    .str.decode("utf-8")
)  

In [ ]:
# Renommer les colonnes pour uniformiser les noms
df_all = df_all.rename(columns={'quand':'date','lieu':'commune'})
df_all.drop(columns=['n'], inplace=True)

,date,commune,departement,prenom,age,contenu_publication,lien_instagram
0,25/11/2022,Prahecq,Deux-Sèvres,Leslie,22.0,NaN,NaN
1,04/01/2023,Perles,Aisne,–,50.0,"Samedi 7 janvier 2023 à Perles (02), le corps ...",https://www.instagram.com/p/CnSIKbRLwTM/?utm_s...
2,31/12/2022,Charleville-Mézières,Ardennes,–,23.0,"Le samedi 31 décembre, une femme de 23 ans a é...",https://www.instagram.com/p/Cm4bp-qLwox/
3,24/12/2022,Saint-Raphaël,Var,Molka,31.0,"Samedi 24 décembre à Saint-Raphaël (83), une f...",https://www.instagram.com/p/Cm2Jl02LGTZ/
4,16/10/2022,Goudelin,Côtes-d'Armor,Jacqueline,67.0,"Dimanche 16 octobre à Goudelin (22), une femme...",https://www.instagram.com/p/Cm1kH5gN24G/


In [ ]:
# Convertir la colonne 'date' en format datetime
df_all['date'] = pd.to_datetime(df_all['date'], format='%d/%m/%Y', errors='coerce')

,date,commune,departement,prenom,age,contenu_publication,lien_instagram
0,2022-11-25,Prahecq,Deux-Sèvres,Leslie,22.0,NaN,NaN
1,2023-01-04,Perles,Aisne,–,50.0,"Samedi 7 janvier 2023 à Perles (02), le corps ...",https://www.instagram.com/p/CnSIKbRLwTM/?utm_s...
2,2022-12-31,Charleville-Mézières,Ardennes,–,23.0,"Le samedi 31 décembre, une femme de 23 ans a é...",https://www.instagram.com/p/Cm4bp-qLwox/
3,2022-12-24,Saint-Raphaël,Var,Molka,31.0,"Samedi 24 décembre à Saint-Raphaël (83), une f...",https://www.instagram.com/p/Cm2Jl02LGTZ/
4,2022-10-16,Goudelin,Côtes-d'Armor,Jacqueline,67.0,"Dimanche 16 octobre à Goudelin (22), une femme...",https://www.instagram.com/p/Cm1kH5gN24G/


In [ ]:
# Normaliser les noms de communes pour faciliter la jointure avec les coordonnées géographiques
df_all['commune_normalise'] = (
    df_all['commune']
    .str.normalize("NFKD")
    .str.encode("ascii", "ignore")
    .str.decode("utf-8")
    .str.lower()
    .str.replace(" ", "-", regex=False)
    .str.replace("'", "-", regex=False)
)

In [ ]:
# Normaliser les noms de départements pour faciliter la jointure avec les coordonnées géographiques
df_all['dep_normalise'] = (
    df_all['departement']
    .str.normalize("NFKD")
    .str.encode("ascii", "ignore")
    .str.decode("utf-8")
    .str.lower()
    .str.replace(" ", "-", regex=False)
    .str.replace("'", "-", regex=False)
)

In [157]:
#suppression des espaces et tiret au début et à la fin de commune_normalise et dep_normalise
df_all['commune_normalise'] = df_all['commune_normalise'].str.strip('- ') 
df_all['dep_normalise'] = df_all['dep_normalise'].str.strip('- ') 

df_all['commune_normalise'] = df_all['commune_normalise'].str.rstrip('- ') 
df_all['dep_normalise'] = df_all['dep_normalise'].str.rstrip('- ') 

df_all.head(100)

,date,commune,departement,prenom,age,contenu_publication,lien_instagram,commune_normalise,dep_normalise
0,2022-11-25,Prahecq,Deux-Sèvres,Leslie,22.0,NaN,NaN,prahecq,deux-sevres
1,2023-01-04,Perles,Aisne,–,50.0,"Samedi 7 janvier 2023 à Perles (02), le corps ...",https://www.instagram.com/p/CnSIKbRLwTM/?utm_s...,perles,aisne
2,2022-12-31,Charleville-Mézières,Ardennes,–,23.0,"Le samedi 31 décembre, une femme de 23 ans a é...",https://www.instagram.com/p/Cm4bp-qLwox/,charleville-mezieres,ardennes
3,2022-12-24,Saint-Raphaël,Var,Molka,31.0,"Samedi 24 décembre à Saint-Raphaël (83), une f...",https://www.instagram.com/p/Cm2Jl02LGTZ/,saint-raphael,var
4,2022-10-16,Goudelin,Côtes-d'Armor,Jacqueline,67.0,"Dimanche 16 octobre à Goudelin (22), une femme...",https://www.instagram.com/p/Cm1kH5gN24G/,goudelin,cotes-d-armor
...,...,...,...,...,...,...,...,...,...
95,2022-05-16,Béziers,Hérault,Claire,38.0,"Lundi 16 mai à Béziers (34), une femme de 38 a...",https://twitter.com/NousToutesOrg/status/15284...,beziers,herault
96,2022-05-09,Saint-Rémy,Saône-et-Loire,Sarah,32.0,"Le 8 mai à Gergy (71), Audrey (33ans) et sa fi...",https://twitter.com/NousToutesOrg/status/15244...,saint-remy,saone-et-loire
97,2022-05-09,Gergy,Saône-et-Loire,Audrey,33.0,"Le 8 mai à Gergy (71), Audrey (33ans) et sa fi...",https://twitter.com/NousToutesOrg/status/15244...,gergy,saone-et-loire
98,2022-05-08,Grézieu-la-Varenne,Rhône,Nathalie,33.0,"Dimanche 8 mai à Grézieu-la-Varenne (69), une ...",https://twitter.com/NousToutesOrg/status/15237...,grezieu-la-varenne,rhone


In [158]:
#dans df_all, remplacer les noms de commune contenant '*e arrondissement' par 'Paris'
df_all['commune_normalise'] = df_all['commune_normalise'].str.replace(r'\d{1,2}e-arrondissement', 'paris', regex=True)
df_all.head(100)

,date,commune,departement,prenom,age,contenu_publication,lien_instagram,commune_normalise,dep_normalise
0,2022-11-25,Prahecq,Deux-Sèvres,Leslie,22.0,NaN,NaN,prahecq,deux-sevres
1,2023-01-04,Perles,Aisne,–,50.0,"Samedi 7 janvier 2023 à Perles (02), le corps ...",https://www.instagram.com/p/CnSIKbRLwTM/?utm_s...,perles,aisne
2,2022-12-31,Charleville-Mézières,Ardennes,–,23.0,"Le samedi 31 décembre, une femme de 23 ans a é...",https://www.instagram.com/p/Cm4bp-qLwox/,charleville-mezieres,ardennes
3,2022-12-24,Saint-Raphaël,Var,Molka,31.0,"Samedi 24 décembre à Saint-Raphaël (83), une f...",https://www.instagram.com/p/Cm2Jl02LGTZ/,saint-raphael,var
4,2022-10-16,Goudelin,Côtes-d'Armor,Jacqueline,67.0,"Dimanche 16 octobre à Goudelin (22), une femme...",https://www.instagram.com/p/Cm1kH5gN24G/,goudelin,cotes-d-armor
...,...,...,...,...,...,...,...,...,...
95,2022-05-16,Béziers,Hérault,Claire,38.0,"Lundi 16 mai à Béziers (34), une femme de 38 a...",https://twitter.com/NousToutesOrg/status/15284...,beziers,herault
96,2022-05-09,Saint-Rémy,Saône-et-Loire,Sarah,32.0,"Le 8 mai à Gergy (71), Audrey (33ans) et sa fi...",https://twitter.com/NousToutesOrg/status/15244...,saint-remy,saone-et-loire
97,2022-05-09,Gergy,Saône-et-Loire,Audrey,33.0,"Le 8 mai à Gergy (71), Audrey (33ans) et sa fi...",https://twitter.com/NousToutesOrg/status/15244...,gergy,saone-et-loire
98,2022-05-08,Grézieu-la-Varenne,Rhône,Nathalie,33.0,"Dimanche 8 mai à Grézieu-la-Varenne (69), une ...",https://twitter.com/NousToutesOrg/status/15237...,grezieu-la-varenne,rhone


In [159]:
#cas particulier de bourgoin-jallieu-/-lyon
df_all['commune_normalise'] = df_all['commune_normalise'].str.replace('bourgoin-jallieu-/-lyon', 'bourgoin-jallieu')

#remplcer dep_normalise de rhone à isere pour bourgoin-jallieu
df_all.loc[df_all['commune_normalise']=='bourgoin-jallieu', 'dep_normalise'] = 'isere'

In [ ]:
#rajout d'une colonne 'meurtrier'
df_all.insert(6, 'meurtrier', np.nan)

## Utilsation de gpt-4o-mini pour compléter la colonne 'meurtrier' en se basant sur 'contenu_publication'

In [ ]:

from openai import OpenAI
client = OpenAI(api_key="")


In [172]:
def extraire_meurtrier(texte):
    prompt = f"""
    Analyse ce texte : "{texte}"
    Dis-moi en un seul mot qui est le meurtrier (par exemple: conjoint, père, fils, voisin, inconnu, locataire ...). 
    Pour les termes conjoint, mari, époux, compagnon, petit-ami, concubin etc, utilise uniquement "conjoint". 
    Mets 'client' si la personne assassinée semble être une prostituée assassinée par un client.
    Si ce n'est pas clair, mets 'inconnu'
    """
    completion = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[{"role": "user", "content": prompt}],
        max_tokens=5,
        temperature=0
    )
    return completion.choices[0].message.content.strip().lower()

In [165]:
df_all["meurtrier"] = df_all["contenu_publication"].apply(extraire_meurtrier)

In [ ]:
df_inconnu = df_all[df_all['meurtrier']=='inconnu']

In [ ]:
#On réapplique pour forcer une nouvelle tentative d'extraction
df_inconnu['meurtrier'] = df_inconnu['contenu_publication'].apply(extraire_meurtrier)

C:\Users\ghirg\AppData\Local\Temp\ipykernel_15872\2089889159.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_inconnu['meurtrier'] = df_inconnu['contenu_publication'].apply(extraire_meurtrier)


In [174]:
#remettre df_inconnu dans df_all
df_all.update(df_inconnu)

## On charge les données des communes

In [ ]:
df_commune = pd.read_csv('../data/raw/communes-france-2025.csv')

C:\Users\ghirg\AppData\Local\Temp\ipykernel_15872\954040633.py:1: DtypeWarning: Columns (1,12,14,16,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_commune = pd.read_csv('../data/raw/communes-france-2025.csv')


,Unnamed: 0,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,...,longitude_mairie,latitude_centre,longitude_centre,grille_densite,grille_densite_texte,niveau_equipements_services,niveau_equipements_services_texte,gentile,url_wikipedia,url_villedereve
0,0,01001,L'Abergement-Clémenciat,Abergement-Clémenciat,à Abergement-Clémenciat,de l'Abergement-Clémenciat,l-abergement-clemenciat,L'ABERGEMENT-CLÉMENCIAT,COM,commune,...,4.921,46.153,4.926,6,Rural à habitat dispersé,0.0,communes non pôle,NaN,https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01001-l-abergemen...
1,1,01002,L'Abergement-de-Varey,Abergement-de-Varey,à Abergement-de-Varey,de l'Abergement-de-Varey,l-abergement-de-varey,L'ABERGEMENT-DE-VAREY,COM,commune,...,5.423,46.009,5.428,6,Rural à habitat dispersé,0.0,communes non pôle,"Abergementais, Abergementaises",https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01002-l-abergemen...
2,2,01004,Ambérieu-en-Bugey,Ambérieu-en-Bugey,à Ambérieu-en-Bugey,d'Ambérieu-en-Bugey,amberieu-en-bugey,AMBÉRIEU-EN-BUGEY,COM,commune,...,5.360,45.961,5.373,2,Centres urbains intermédiaires,3.0,centres structurants d'équipements et de services,"Ambarrois, Ambarroises",https://fr.wikipedia.org/wiki/fr:Ambérieu-en-B...,https://villedereve.fr/ville/01004-amberieu-en...
3,3,01005,Ambérieux-en-Dombes,Ambérieux-en-Dombes,à Ambérieux-en-Dombes,d'Ambérieux-en-Dombes,amberieux-en-dombes,AMBÉRIEUX-EN-DOMBES,COM,commune,...,4.903,45.996,4.912,5,Bourgs ruraux,1.0,centres locaux d'équipements et de services,Ambarrois,https://fr.wikipedia.org/wiki/fr:Ambérieux-en-...,https://villedereve.fr/ville/01005-amberieux-e...
4,4,01006,Ambléon,Ambléon,à Ambléon,d'Ambléon,ambleon,AMBLÉON,COM,commune,...,5.601,45.750,5.594,6,Rural à habitat dispersé,0.0,communes non pôle,Ambléonais,https://fr.wikipedia.org/wiki/fr:Ambléon,https://villedereve.fr/ville/01006-ambleon


In [176]:
#Sélection des colonnes utiles'
df_commune = df_commune[['nom_sans_accent','dep_nom','dep_code','population','superficie_km2','densite','latitude_mairie','longitude_mairie']]

In [177]:
#créer commune normalisée (sans accent, minuscule, avec tiret à la place des espaces et des apostrophes)
df_commune['commune_normalise'] = (
    df_commune['nom_sans_accent']
    .str.normalize("NFKD")
    .str.encode("ascii", "ignore")
    .str.decode("utf-8")
    .str.lower()
    .str.replace(" ", "-", regex=False)
    .str.replace("'", "-", regex=False)
)

df_commune['dep_normalise'] = (
    df_commune['dep_nom']
    .str.normalize("NFKD")
    .str.encode("ascii", "ignore")
    .str.decode("utf-8")
    .str.lower()
    .str.replace(" ", "-", regex=False)
    .str.replace("'", "-", regex=False)
)

In [179]:
#inner join entre df_all et df_commune sur les colonnes 'commune_sans_accents' et 'departement_sans_accents' de df_all et 'nom_sans_accent' et 'dep_sans_accents' de df_commune
df_merged = pd.merge(df_all, 
                     df_commune, 
                     how='left', 
                     left_on=['commune_normalise','dep_normalise'], 
                     right_on=['commune_normalise','dep_normalise'])

In [181]:
df_merged.rename(columns={'latitude_mairie': 'latitude', 'longitude_mairie': 'longitude'}, inplace=True)

In [182]:
 #enlever les lignes où com_norm est NaN
df_merged_sans_nan = df_merged[~df_merged['dep_code'].isna()]
df_merged_sans_nan.info()

<class 'pandas.core.frame.DataFrame'>
Index: 475 entries, 0 to 527
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 475 non-null    datetime64[ns]
 1   commune              475 non-null    object        
 2   departement          475 non-null    object        
 3   prenom               474 non-null    object        
 4   age                  473 non-null    float64       
 5   contenu_publication  474 non-null    object        
 6   meurtrier            475 non-null    object        
 7   lien_instagram       474 non-null    object        
 8   commune_normalise    475 non-null    object        
 9   dep_normalise        475 non-null    object        
 10  nom_sans_accent      475 non-null    object        
 11  dep_nom              475 non-null    object        
 12  dep_code             475 non-null    object        
 13  population           475 non-null    flo

In [183]:
df_merged_nan = df_merged[df_merged['dep_code'].isna()]
df_merged_nan.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54 entries, 1 to 528
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 54 non-null     datetime64[ns]
 1   commune              54 non-null     object        
 2   departement          54 non-null     object        
 3   prenom               54 non-null     object        
 4   age                  54 non-null     float64       
 5   contenu_publication  54 non-null     object        
 6   meurtrier            54 non-null     object        
 7   lien_instagram       54 non-null     object        
 8   commune_normalise    54 non-null     object        
 9   dep_normalise        54 non-null     object        
 10  nom_sans_accent      0 non-null      object        
 11  dep_nom              0 non-null      object        
 12  dep_code             0 non-null      object        
 13  population           0 non-null      floa

In [ ]:
#Utilisation de geopy pour récupérer les coordonnées des communes manquantes
import time
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="geoapi_commune")

def get_coords(commune):
    query = f"{commune} "
    try:
        location = geolocator.geocode(query, timeout=5)  # délai plus long
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        print(f"Erreur pour {query}: {e}")
    return None, None

# Pour stocker les résultats déjà trouvés
cache = {}

for commune in df_merged_nan['commune']:
    key = commune
    if key not in cache:  # pas encore cherché
        lat, lon = get_coords(commune)
        cache[key] = (lat, lon)
        time.sleep(1)  # respecter la limite Nominatim
    else:
        lat, lon = cache[key]
    
    df_merged_nan.loc[
        (df_merged_nan['commune'] == commune),
        ['latitude', 'longitude']
    ] = [lat, lon]

In [187]:
#ajouter les lignes de df_merged_nan où latitude n'est pas NaN à df_merged_sans_nan
idx_non_nan = df_merged_nan[~df_merged_nan['latitude'].isna()].index
df_merged_sans_nan = pd.concat([df_merged_sans_nan, df_merged_nan.loc[idx_non_nan]], ignore_index=True)

In [188]:
#correction manuelle des noms de communes_sans_accent mal orthographiés
df_merged_nan.loc[(df_merged_nan['nom_sans_accent']=='amneville-les-thermes'),'nom_sans_accent'] = 'amneville'
df_merged_nan.loc[(df_merged_nan['nom_sans_accent']=='sainte-maure-de-peyrac'),'nom_sans_accent'] = 'sainte-maure-de-peyriac'
df_merged_nan.loc[(df_merged_nan['nom_sans_accent']=='equilivaz/la-salle'),'nom_sans_accent'] = 'la-salle-italia'
df_merged_nan.loc[(df_merged_nan['nom_sans_accent']=='boucaya'),'nom_sans_accent'] = 'bacouya'
df_merged_nan.loc[(df_merged_nan['nom_sans_accent']=='coulange-les-nevers'),'nom_sans_accent'] = 'coulanges-les-nevers'

#corection des communes mal orthographiées
df_merged_nan.loc[(df_merged_nan['commune']=='Amnéville-les-Thermes'),'commune'] = 'Amnéville'
df_merged_nan.loc[(df_merged_nan['commune']=='Sainte-Maure-de-Peyrac'),'commune'] = 'Sainte-Maure-de-Peyriac'
df_merged_nan.loc[(df_merged_nan['commune']=='Equilivaz/La Salle'),'commune'] = 'La Salle-Italia'
df_merged_nan.loc[(df_merged_nan['commune']=='Boucaya'),'commune'] = 'Bacouya'
df_merged_nan.loc[(df_merged_nan['commune']=='Coulange-lès-Nevers'),'commune'] = 'Coulanges-lès-Nevers'

In [189]:
cache = {}
for commune in df_merged_nan['commune']:
    key = commune
    if key not in cache:  # pas encore cherché
        lat, lon = get_coords(commune)
        cache[key] = (lat, lon)
        time.sleep(1)  # respecter la limite Nominatim
    else:
        lat, lon = cache[key]
    
    df_merged_nan.loc[
        (df_merged_nan['commune'] == commune),
        ['latitude', 'longitude']
    ] = [lat, lon]

In [190]:

df_merged_sans_nan = pd.concat([df_merged_sans_nan,df_merged_nan], ignore_index=True)
df_merged_sans_nan.sort_values(by='date', inplace=True)

In [205]:
#repartir meurtrier en nouvelle caégorie : 'conjoint', 'famille autre', 'inconnu', 'client', 'voisin', 'autre'
df_merged_sans_nan['meurtrier_cat'] = df_merged_sans_nan['meurtrier'].replace({
    'conjoint': 'conjoint',
    'frère': 'famille autre',
    'petit-fils': 'famille autre',
    'beau-frère': 'famille autre',
    'ami': 'connu autre',
    'ancien patient': 'connu autre',
    'locataire': 'connu autre',
    'connaissance': 'connu autre',
    'associé': 'connu autre',
    'inconnu': 'inconnu',
    'client': 'client',
    'voisin': 'voisin',
    'oncle': 'famille autre',
    'neveu': 'famille autre',
    'élève': 'connu autre',
})

In [206]:
#sauvegarder df_merged_sans_nan dans un fichier csv
df_merged_sans_nan.to_csv('../data/processed/feminicide_2022_2025.csv', index=False)

# Créer fichier csv pour les statistiques par département

In [210]:
#df avec le total des feminicide par département et la population totale
df_total_dep = df_all.groupby("departement").size().reset_index(name='total_feminicide')
df_total_dep.head()

,departement,total_feminicide
0,Ain,2
1,Aisne,5
2,Alpes-Maritimes,10
3,Alpes-de-Haute-Provence,1
4,Ardennes,4


In [211]:
#calculer la population totale par département
df_dep = df_commune.groupby(["dep_nom","dep_code"])["population"].sum().reset_index()
df_dep = df_dep.groupby("dep_nom")["population"].sum().reset_index()
df_dep.head()

,dep_nom,population
0,Ain,663202
1,Aisne,527468
2,Allier,334872
3,Alpes-Maritimes,1103941
4,Alpes-de-Haute-Provence,166077


In [212]:
#merge entre df_dep et df_total_dep
df_total_dep = pd.merge(df_dep, df_total_dep, left_on="dep_nom", right_on="departement", how="left")
df_total_dep.drop(columns=['departement'], inplace=True)
df_total_dep.head()

,dep_nom,population,total_feminicide
0,Ain,663202,2.0
1,Aisne,527468,5.0
2,Allier,334872,NaN
3,Alpes-Maritimes,1103941,10.0
4,Alpes-de-Haute-Provence,166077,1.0


In [213]:
#calculer le taux de feminicide pour 100k habitants par département
nb_annees = len(years)
df_total_dep['feminicide_per_100k'] = (df_total_dep['total_feminicide'] / df_total_dep['population']) * (100000 / nb_annees)
df_total_dep.head()

,dep_nom,population,total_feminicide,feminicide_per_100k
0,Ain,663202,2.0,0.075392
1,Aisne,527468,5.0,0.236981
2,Allier,334872,NaN,NaN
3,Alpes-Maritimes,1103941,10.0,0.226461
4,Alpes-de-Haute-Provence,166077,1.0,0.150533


In [214]:
df_total_dep["Légende"] = [
    f"{ligne.dep_nom} : pas de données" if pd.isna(ligne.total_feminicide)
    else f"{ligne.dep_nom} : {ligne.feminicide_per_100k:.2f} pour 100 000 habitants par an"
    for _, ligne in df_total_dep.iterrows()
]

In [215]:
df_total_dep.head()

,dep_nom,population,total_feminicide,feminicide_per_100k,Légende
0,Ain,663202,2.0,0.075392,Ain : 0.08 pour 100 000 habitants par an
1,Aisne,527468,5.0,0.236981,Aisne : 0.24 pour 100 000 habitants par an
2,Allier,334872,NaN,NaN,Allier : pas de données
3,Alpes-Maritimes,1103941,10.0,0.226461,Alpes-Maritimes : 0.23 pour 100 000 habitants ...
4,Alpes-de-Haute-Provence,166077,1.0,0.150533,Alpes-de-Haute-Provence : 0.15 pour 100 000 ha...


In [218]:
df_total_dep['dep_nom'].unique()

array(['Ain', 'Aisne', 'Allier', 'Alpes-Maritimes',
       'Alpes-de-Haute-Provence', 'Ardennes', 'Ardèche', 'Ariège', 'Aube',
       'Aude', 'Aveyron', 'Bas-Rhin', 'Bouches-du-Rhône', 'Calvados',
       'Cantal', 'Charente', 'Charente-Maritime', 'Cher', 'Corrèze',
       'Corse-du-Sud', 'Creuse', "Côte-d'Or", "Côtes-d'Armor",
       'Deux-Sèvres', 'Dordogne', 'Doubs', 'Drôme', 'Essonne', 'Eure',
       'Eure-et-Loir', 'Finistère', 'Gard', 'Gers', 'Gironde',
       'Guadeloupe', 'Guyane', 'Haut-Rhin', 'Haute-Corse',
       'Haute-Garonne', 'Haute-Loire', 'Haute-Marne', 'Haute-Savoie',
       'Haute-Saône', 'Haute-Vienne', 'Hautes-Alpes', 'Hautes-Pyrénées',
       'Hauts-de-Seine', 'Hérault', 'Ille-et-Vilaine', 'Indre',
       'Indre-et-Loire', 'Isère', 'Jura', 'La Réunion', 'Landes',
       'Loir-et-Cher', 'Loire', 'Loire-Atlantique', 'Loiret', 'Lot',
       'Lot-et-Garonne', 'Lozère', 'Maine-et-Loire', 'Manche', 'Marne',
       'Martinique', 'Mayenne', 'Mayotte', 'Meurthe-et-Moselle',

In [219]:
#on sépare metropole et dom-tom
df_total_dep_metropole = df_total_dep[~df_total_dep['dep_nom'].isin(['Guadeloupe', 'Martinique', 'Guyane', 'La Réunion', 'Mayotte'])]
df_total_dep_tom = df_total_dep[df_total_dep['dep_nom'].isin(['Guadeloupe', 'Martinique', 'Guyane', 'La Réunion', 'Mayotte'])]

In [220]:
#sauvegarder df_total_dep dans un fichier csv
df_total_dep_metropole.to_csv('../data/processed/df_total_dep.csv', index=False)

## On crée le fichier geojson pour les départements avec les légendes

In [2]:
# On charge les données géographiques pour le bord des départements
import geopandas as gpd
json_dep_url = "https://france-geojson.gregoiredavid.fr/repo/departements.geojson"

json_dep = gpd.read_file(json_dep_url)  
json_dep.sort_values(by='nom', inplace=True)
json_dep.reset_index(drop=True, inplace=True)
json_dep.head()

,code,nom,geometry
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.78024 46.18905, ..."
1,02,Aisne,"POLYGON ((3.1727 50.012, 3.1822 50.01234, 3.21..."
2,03,Allier,"POLYGON ((3.03206 46.79491, 3.03684 46.7844, 3..."
3,06,Alpes-Maritimes,"MULTIPOLYGON (((7.06712 43.51365, 7.03613 43.5..."
4,04,Alpes-de-Haute-Provence,"POLYGON ((5.67604 44.19143, 5.69209 44.18648, ..."


In [5]:
# On crée le fichier geojson pour les départements avec les légendes
df_total_dep_metropole = pd.read_csv('../data/processed/df_total_dep.csv')
json_dep = pd.concat([json_dep, df_total_dep_metropole[["Légende"]]], axis=1)

In [6]:
#sauver json_dep dans un fichier geojson
json_dep.to_file("../data/processed/departements_feminicide.geojson", driver='GeoJSON')